In [1]:
#Parameters
model_name = "retfound_green"
run_id = 0

In [2]:
# Multi image classification
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # allow multiple libiomp5md
os.environ["OMP_NUM_THREADS"] = "1"           # keep OpenMP under control

import torch
import albumentations as A
import numpy as np
import pandas as pd
from model import UnifiedBackbone
from fundus_dataset import FundusDataset
from torch.utils.data import DataLoader

C:\ProgramData\Anaconda3\envs\mBRSETnew\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
aug_type = 0
if model_name != "retfound_green":
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.485, 0.456, 0.406), 
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
else:
    train_tf = A.Compose([
        A.RandomResizedCrop(height=392, width=392, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    
    val_tf = A.Compose([
        A.Resize(392, 392),
        A.Normalize(mean=(0.5, 0.5, 0.5), 
                    std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])

In [5]:
from diagnosis_train_eval import train_model, validate, evaluate_thresholds
from model import UnifiedBackboneMultiConcat,UnifiedBackboneMulti
from torch.utils.data import DataLoader
import numpy as np
import torch
import torch.nn as nn

def compute_class_weights(df, threshold=0):
    labels = (df["final_icdr"] > threshold).astype(int).values
    class_counts = np.bincount(labels)
    weights = 1.0 / class_counts
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float32)
    

device = "cuda"

In [6]:
data_dir = r'C:\\Users\\preet\\Documents\\mBRSET\\mBRSET_image_quality\\data\\'

train_df_single = pd.read_pickle(data_dir + "mbrset_icdr_quality_2826_train_full.pkl")
val_df_single = pd.read_pickle(data_dir + "mbrset_icdr_quality_2826_val_full.pkl")
test_df_single = pd.read_pickle(data_dir + "mbrset_icdr_quality_2826_test_full.pkl")

train_df_single.drop(columns = ["Unnamed: 0"], inplace=True)
val_df_single.drop(columns = ["Unnamed: 0"], inplace=True)
test_df_single.drop(columns = ["Unnamed: 0"], inplace=True)

train_df_single.reset_index(drop=True, inplace=True)
val_df_single.reset_index(drop=True, inplace=True)
test_df_single.reset_index(drop=True, inplace=True)


In [209]:
img_root = r"C:\\Users\\preet\\Documents\\mBRSET\\mbrset-a-mobile-brazilian-retinal-dataset-1.0\\images"
train_ds_single = FundusDataset(train_df_single,img_root, high_quality_tf=train_tf, low_quality_tf=train_tf, label_col='final_icdr')
val_ds_single = FundusDataset(val_df_single,img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col='final_icdr')
test_ds_single = FundusDataset(test_df_single,img_root, high_quality_tf=val_tf, low_quality_tf=val_tf, label_col='final_icdr')

In [211]:

train_loader = DataLoader(train_ds_single, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds_single,   batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds_single,  batch_size=8, shuffle=False, num_workers=2)


In [212]:
import torch
from model import UnifiedBackbone
from diagnosis_train_eval import train_model, validate, evaluate_thresholds 

loss_fn = torch.nn.CrossEntropyLoss()
device = 'cuda'
ba_target = 0.9

# Load trained diagnosis backbone
best_model = UnifiedBackbone(model_name=model_name).cuda().float()
best_model.load_state_dict(torch.load("mBRSET_img_diagnosis_model_392.pth"))

# Run validation
loss1, metrics_val, all_files, corr_files, incorrect_files = validate(
    best_model.float().cuda(), val_loader, loss_fn, device,
    test=True, ba_target=ba_target
)   #REMOVE BA TARGET???? 

# Run test evaluation
loss1, metrics_test, all_files, corr_files, incorrect_files = validate(
    best_model.float().cuda(), test_loader, loss_fn, device,
    test=True, ba_target=ba_target
)

# Attach image-level probabilities to validation set
df_probs_val = pd.DataFrame({
    "file": all_files,
    "prob": metrics_val["all_probs"]
})
df_val_eval = val_df_single.merge(df_probs_val, on="file", how="left")

# Attach image-level probabilities to test set
df_probs_test = pd.DataFrame({
    "file": all_files,
    "prob": metrics_test["all_probs"]
})
df_test_eval = test_df_single.merge(df_probs_test, on="file", how="left")

Val: 100%|█████████████████████████████████████████████████████████████████████████████| 84/84 [00:20<00:00,  4.05it/s]


Best T1, T2: 0.4893 0.503
Coverage: 0.9970059880239521
Balanced Accuracy: 0.9358194560134694
Confusion Matrix (rows=true, cols=pred):
[[483  32]
 [ 10 141]]
T1 = 0.4893, T2 = 0.5029
Balanced Accuracy = 0.9358
Coverage = 99.70% of samples accepted


Val: 100%|█████████████████████████████████████████████████████████████████████████████| 79/79 [00:17<00:00,  4.41it/s]


Best T1, T2: 0.3027 0.682
Coverage: 0.8487261146496815
Balanced Accuracy: 0.9002333372757719
Confusion Matrix (rows=true, cols=pred):
[[393  18]
 [ 19 103]]
T1 = 0.3027, T2 = 0.6821
Balanced Accuracy = 0.9002
Coverage = 84.87% of samples accepted


In [214]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# Core constraints
num_imgs = 1
MIN_COVERAGE = 0.50
MIN_SENSITIVITY = 0.7

POOLING_METHOD = "mean"

# Fixed image-level confidence margins - atleast 3 images should be kept for every patient in mBRSET
t_low  = 0.5 - 0.079
t_high = 0.5 + 0.079
min_conf_imgs = 1

# Patient-level margins to sweep
patient_margins = [0.20, 0.25, 0.30, 0.35, 0.40]

results = []

from multi_image_val_test import compute_perf_metrics


df_val_eval = df_val_eval.sort_values(["patient", "file"])

for pat_margin in patient_margins:

    # Patient-level decision thresholds
    p_low  = 0.5 - pat_margin
    p_high = 0.5 + pat_margin

    coverage, ba, sens, spec, df_output = compute_perf_metrics(
        df_val_eval, t_low, t_high,
        min_conf_imgs, p_low, p_high,
        num_imgs, "mBRSET"
    )

    if coverage >= MIN_COVERAGE and sens >= MIN_SENSITIVITY:
        results.append({
            "img_margin": img_margin,
            "t_low": t_low,
            "t_high": t_high,
            "patient_margin": pat_margin,
            "p_low": p_low,
            "p_high": p_high,
            "min_conf_imgs": min_conf_imgs,
            "coverage": coverage,
            "BA": ba,
            "sensitivity": sens,
            "specificity": spec
        })

results = pd.DataFrame(results)

In [215]:
from multi_image_val_test import determine_test_params
df_test_params = determine_test_params(results)

   img_margin  t_low  t_high  patient_margin  p_low  p_high  min_conf_imgs   
0        0.10   0.40    0.60            0.20   0.30    0.70              1  \
1        0.10   0.40    0.60            0.25   0.25    0.75              1   
2        0.10   0.40    0.60            0.30   0.20    0.80              1   
3        0.10   0.40    0.60            0.35   0.15    0.85              1   
4        0.10   0.40    0.60            0.40   0.10    0.90              1   
5        0.15   0.35    0.65            0.20   0.30    0.70              1   
6        0.15   0.35    0.65            0.25   0.25    0.75              1   
7        0.15   0.35    0.65            0.30   0.20    0.80              1   
8        0.15   0.35    0.65            0.35   0.15    0.85              1   
9        0.15   0.35    0.65            0.40   0.10    0.90              1   

   coverage        BA  sensitivity  specificity  
0  0.898204  0.945906     0.944444     0.947368  
1  0.832335  0.947390     0.942857     0.

,coverage_bin,coverage,BA,img_margin,t_low,t_high,patient_margin,p_low,p_high,min_conf_imgs
0,0.85,0.898204,0.945906,0.1,0.4,0.6,0.20,0.30,0.70,1
1,0.80,0.832335,0.947390,0.1,0.4,0.6,0.25,0.25,0.75,1
2,0.75,0.766467,0.963796,0.1,0.4,0.6,0.30,0.20,0.80,1
3,0.65,0.658683,0.958734,0.1,0.4,0.6,0.35,0.15,0.85,1
4,0.55,0.556886,0.975806,0.1,0.4,0.6,0.40,0.10,0.90,1


In [216]:
test_results = []
for i in range(len(df_test_params)):
    row = df_test_params.iloc[i]
    print(row["t_low"], row["t_high"], row["p_low"], row["p_high"], row["min_conf_imgs"])
    coverage, ba, sens, spec, df_output = compute_perf_metrics(df_test_eval, row["t_low"], row["t_high"], row["min_conf_imgs"], row["p_low"], row["p_high"], num_imgs, "mBRSET")
    test_results.append({
        "t_low": row["t_low"],
        "t_high":  row["t_high"],
        "p_low":  row["p_low"],
        "p_high": row["p_high"],
        "min_conf_imgs":  row["min_conf_imgs"],
        "coverage": coverage,
        "BA": ba,
        "sensitivity": sens,
        "specificity": spec
    })

#add 100% coverage point
coverage, ba, sens, spec, df_output = compute_perf_metrics(df_test_eval, 0.5, 0.5, 0, 0.5, 0.5, num_imgs, "mBRSET") 
test_results.append({
    "t_low": 0.5,
    "t_high":  0.5,
    "p_low":  0.5,
    "p_high": 0.5,
    "min_conf_imgs":  0,
    "coverage": coverage,
    "BA": ba,
    "sensitivity": sens,
    "specificity": spec
})
test_results = pd.DataFrame(test_results)

0.4 0.6 0.3 0.7 1.0
0.4 0.6 0.25 0.75 1.0
0.4 0.6 0.2 0.8 1.0
0.4 0.6 0.15000000000000002 0.85 1.0
0.4 0.6 0.09999999999999998 0.9 1.0


,t_low,t_high,p_low,p_high,min_conf_imgs,coverage,BA,sensitivity,specificity
0,0.4,0.6,0.30,0.70,1.0,0.821656,0.901515,0.833333,0.969697
1,0.4,0.6,0.25,0.75,1.0,0.757962,0.897126,0.827586,0.966667
2,0.4,0.6,0.20,0.80,1.0,0.687898,0.921928,0.880000,0.963855
3,0.4,0.6,0.15,0.85,1.0,0.617834,0.933056,0.880000,0.986111
4,0.4,0.6,0.10,0.90,1.0,0.458599,0.946318,0.913043,0.979592
5,0.5,0.5,0.50,0.50,0.0,1.000000,0.862984,0.823529,0.902439


In [184]:
data_result = r"C:\Users\preet\Documents\mBRSET\mBRSET_image_quality\results\mbrset_results"
test_results.to_csv(data_result + "\FINAL_"+str(num_imgs)+ "_mBRSET" + str(model_name) + "_21426_" + str(run_id) + ".csv")

## Choosing mBRSET images for Demo

In [220]:
patient_probs = df_test_eval.groupby("patient")["prob"].apply(list)
maxthres = 0.8
minthres = 0.2
patient_prob_dict = {
    patient: probs
    for patient, probs in patient_probs.items()
    if max(probs) > maxthres and min(probs) < minthres
}

patient_prob_dict


{419: [0.97802734375, 0.95947265625, 0.07684326171875, 0.1517333984375],
 662: [0.1695556640625, 0.039154052734375, 0.6748046875, 0.81787109375],
 750: [0.998046875, 0.302734375, 0.0775146484375, 0.2249755859375],
 817: [0.8349609375, 0.161865234375, 0.050750732421875, 0.051544189453125],
 883: [0.07781982421875, 0.493896484375, 0.82470703125, 0.716796875],
 943: [0.1900634765625, 0.84130859375, 0.42626953125, 0.43408203125],
 1027: [0.08843994140625, 0.892578125, 0.94287109375, 0.8974609375],
 1050: [0.96630859375, 0.99853515625, 0.07940673828125, 0.97705078125],
 1083: [0.80615234375, 0.48388671875, 0.5078125, 0.1593017578125],
 1219: [0.83203125, 0.57568359375, 0.6005859375, 0.1026611328125]}

In [221]:
# group probs per patient
patient_probs = df_test_eval.groupby("patient")["prob"].apply(list)

# check patient label = 0 (all images)
patient_labels = df_test_eval.groupby("patient")["final_icdr"].apply(set)

maxthres = 0.8
minthres = 0.2

patient_prob_dict = {
    patient: probs
    for patient, probs in patient_probs.items()
    if (
        len(probs) == 4
        and sum(minthres <= p <= maxthres for p in probs) == 2   # two between 0.2 and 0.8
        and sum(p < minthres for p in probs) == 2                # two below 0.2
        and patient_labels[patient] == {0}                       # label is zero
    )
}

patient_prob_dict


{128: [0.1480712890625, 0.29443359375, 0.114501953125, 0.2744140625],
 211: [0.476318359375, 0.281494140625, 0.0343017578125, 0.0433349609375],
 511: [0.07958984375, 0.32275390625, 0.06658935546875, 0.2095947265625],
 579: [0.144287109375, 0.037994384765625, 0.328369140625, 0.370361328125],
 587: [0.26806640625, 0.6728515625, 0.02044677734375, 0.04925537109375],
 705: [0.042816162109375, 0.08935546875, 0.250732421875, 0.449462890625],
 722: [0.426025390625, 0.053863525390625, 0.046783447265625, 0.6806640625],
 823: [0.1256103515625, 0.176025390625, 0.283447265625, 0.444580078125],
 856: [0.41259765625, 0.1778564453125, 0.66748046875, 0.08502197265625],
 926: [0.27685546875, 0.12445068359375, 0.07464599609375, 0.411376953125],
 958: [0.260009765625, 0.445068359375, 0.177001953125, 0.11273193359375],
 972: [0.09503173828125, 0.11492919921875, 0.46240234375, 0.310791015625],
 1152: [0.0675048828125, 0.393798828125, 0.1337890625, 0.2587890625],
 1224: [0.139404296875, 0.25927734375, 0.1280

In [222]:
df_test_eval[df_test_eval["patient"] == 1152].head()

,patient,file,final_icdr,final_quality,laterality,prob
552,1152,1152.1.jpg,False,1,True,0.067505
553,1152,1152.2.jpg,False,1,True,0.393799
554,1152,1152.3.jpg,False,1,False,0.133789
555,1152,1152.4.jpg,False,1,False,0.258789


In [202]:
df_val_eval[df_val_eval["patient"]>160].head(50)

,patient,file,final_icdr,final_quality,laterality,prob
84,195,195.1.jpg,False,1,True,0.995117
85,195,195.2.jpg,False,1,True,0.987793
86,195,195.3.jpg,False,1,False,0.341797
87,195,195.4.jpg,False,1,False,0.379150
88,197,197.1.jpg,False,1,True,0.214111
89,197,197.2.jpg,False,1,True,0.600586
90,197,197.3.jpg,False,1,False,0.129883
91,197,197.4.jpg,False,1,False,0.068054
92,198,198.1.jpg,False,1,True,0.028976
93,198,198.2.jpg,False,1,True,0.269531
